# Huấn luyện mô hình thiệt chẩn (bản thử nghiệm)

**Trước khi chạy:**
1. Tải file `shezhen_datasets1.zip` (bộ TCM-Tongue) lên **Google Drive**, để ở thư mục gốc *My Drive*.
2. Bật GPU: menu **Runtime → Change runtime type → T4 GPU → Save**.
3. Chạy lần lượt từng ô từ trên xuống (bấm nút ▶ bên trái mỗi ô, hoặc Shift+Enter).

**Kết quả cuối:** file `model.onnx` (khoảng 10 MB) nằm trong Drive, thư mục `thiet_chan`. File này là thứ đưa lên trang web.

Bộ dữ liệu là dữ liệu công khai (CC0), không phải ảnh bệnh nhân của phòng khám, nên dùng Colab/Drive cho bước này là phù hợp.

## 1. Kiểm tra GPU
Nếu báo lỗi *command not found* thì chưa bật GPU, quay lại bước 2 ở trên.

In [ ]:
!nvidia-smi

## 2. Kết nối Google Drive
Sẽ hiện cửa sổ xin quyền, chọn tài khoản Google đã chứa file zip. Nếu anh để file zip ở thư mục khác, sửa dòng `ZIP_PATH`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP_PATH = '/content/drive/MyDrive/shezhen_datasets1.zip'   # sửa nếu để chỗ khác
OUT_DIR  = '/content/drive/MyDrive/thiet_chan'               # nơi lưu kết quả
assert os.path.exists(ZIP_PATH), f'Không thấy file {ZIP_PATH} — kiểm tra lại tên và vị trí file zip trên Drive'
os.makedirs(OUT_DIR, exist_ok=True)
print('OK, đã thấy file zip')

## 3. Giải nén
Chỉ lấy thư mục `shezhenv3-txt` (định dạng YOLO). Hai thư mục coco/xml chứa cùng bộ ảnh nên bỏ qua. Mất vài phút.

In [ ]:
!unzip -q -o "{ZIP_PATH}" "*shezhenv3-txt/*" -d /content/data
!du -sh /content/data

## 4. Kiểm tra dữ liệu và tạo file cấu hình
Bước này sửa hai lỗi đã phát hiện trong bộ gốc:
- File cấu hình gốc trỏ về ổ `E:` trên máy tác giả → tạo lại file mới.
- Danh sách nhãn lệch giữa các định dạng (20 hay 21 nhãn) → kiểm tra mã nhãn thực tế trong file nhãn YOLO. Nếu có mã ≥ 20, ô này sẽ dừng và báo.

In [ ]:
import glob, collections, yaml
import pandas as pd

roots = glob.glob('/content/data/**/shezhenv3-txt', recursive=True)
assert roots, 'Không tìm thấy thư mục shezhenv3-txt sau khi giải nén'
ROOT = roots[0]

# 20 nhãn theo file cấu hình YOLO gốc (mã 0–19)
NAMES = ['jiankangshe','botaishe','hongshe','zishe','pangdashe','shoushe','hongdianshe',
         'liewenshe','chihenshe','baitaishe','huangtaishe','heitaishe','huataishe',
         'shenquao','shenqutu','gandanao','gandantu','piweiao','xinfeitu','xinfeiao']
VI = ['Lưỡi bình thường','Rêu bong','Lưỡi đỏ','Lưỡi tím','Lưỡi bệu','Lưỡi gầy mỏng','Lưỡi có điểm đỏ',
      'Lưỡi nứt','Lưỡi có hằn răng','Rêu trắng','Rêu vàng','Rêu đen','Rêu trơn ướt',
      'Vùng thận lõm','Vùng thận lồi','Vùng can đởm lõm','Vùng can đởm lồi','Vùng tỳ vị lõm',
      'Vùng tâm phế lồi','Vùng tâm phế lõm']

IMG_EXT = ('.jpg','.jpeg','.png','.bmp','.webp')
table = collections.defaultdict(dict)
problems = []
for split in ['train','val','test']:
    img_dir, lab_dir = f'{ROOT}/{split}/images', f'{ROOT}/{split}/labels'
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(IMG_EXT)]
    labs = {os.path.splitext(f)[0] for f in os.listdir(lab_dir) if f.endswith('.txt')}
    no_label = [f for f in imgs if os.path.splitext(f)[0] not in labs]
    cnt = collections.Counter()
    for l in labs:
        for line in open(f'{lab_dir}/{l}.txt', encoding='utf-8', errors='ignore'):
            p = line.split()
            if not p: continue
            c = int(float(p[0]))
            cnt[c] += 1
            if not 0 <= c < len(NAMES):
                problems.append((split, l, c))
    print(f'{split:5s}: {len(imgs):5d} ảnh | {len(no_label):4d} ảnh không có file nhãn')
    for c in range(len(NAMES)):
        table[c][split] = cnt.get(c, 0)

df = pd.DataFrame([{'Mã': c, 'Tên gốc': NAMES[c], 'Tiếng Việt': VI[c], **table[c]} for c in range(len(NAMES))])
display(df)

if problems:
    raise ValueError(f'Có {len(problems)} nhãn mang mã ngoài 0–19, ví dụ {problems[:5]}. '
                     'Bộ nhãn có thể theo danh sách 21 nhãn — cần đối chiếu lại trước khi huấn luyện.')

data = {'path': ROOT, 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
        'names': dict(enumerate(NAMES))}
with open('/content/data.yaml', 'w') as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)
df.to_csv(f'{OUT_DIR}/thong_ke_nhan.csv', index=False)
print('Đã tạo /content/data.yaml — dữ liệu hợp lệ, sang bước tiếp theo.')

## 5. Cài thư viện huấn luyện

In [ ]:
!pip -q install ultralytics

## 6. Huấn luyện
Dùng mô hình nhỏ nhất (YOLO11n) để chạy nhanh trên điện thoại. 60 vòng học trên GPU T4 mất khoảng 1–2 giờ.
Kết quả lưu thẳng vào Drive, nên nếu Colab bị ngắt giữa chừng anh không mất công: chạy ô **6b** để học tiếp.
Để tab trình duyệt mở trong lúc chạy.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/data.yaml', epochs=60, imgsz=640, batch=16, patience=15,
            project=OUT_DIR, name='v1', exist_ok=True)

### 6b. Chỉ chạy ô này nếu Colab bị ngắt giữa lúc huấn luyện
(Chạy lại các ô 2, 3, 4, 5 trước, rồi bỏ dấu `#` ở hai dòng dưới và chạy ô này thay cho ô 6.)

In [ ]:
# from ultralytics import YOLO
# YOLO(f'{OUT_DIR}/v1/weights/last.pt').train(resume=True)

## 7. Đánh giá trên tập kiểm tra (553 ảnh máy chưa từng thấy)
**AP50** là thước đo độ chính xác khi khoanh vùng: 1,0 là hoàn hảo, 0,5 là trung bình. Nhãn nào dưới 0,4 thì không nên hiển thị cho người dùng.

In [ ]:
from ultralytics import YOLO
best = YOLO(f'{OUT_DIR}/v1/weights/best.pt')
m = best.val(data='/content/data.yaml', split='test', imgsz=640)

rows = []
for i, c in enumerate(m.box.ap_class_index):
    c = int(c)
    rows.append({'Mã': c, 'Đặc điểm': VI[c],
                 'AP50': round(float(m.box.ap50[i]), 3),
                 'AP50-95': round(float(m.box.ap[i]), 3)})
res = pd.DataFrame(rows).sort_values('AP50', ascending=False)
display(res)
res.to_csv(f'{OUT_DIR}/ket_qua_test.csv', index=False)
print('AP50 trung bình:', round(float(m.box.map50), 3))

## 8. Xuất file mô hình cho trang web
File `model.onnx` sẽ được lưu vào Drive (`thiet_chan/model.onnx`) và tự tải về máy.

In [ ]:
import shutil
path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
shutil.copy(path, f'{OUT_DIR}/model.onnx')
print('Kích thước:', round(os.path.getsize(f'{OUT_DIR}/model.onnx') / 1e6, 1), 'MB')

from google.colab import files
files.download(f'{OUT_DIR}/model.onnx')